ENTRY 2. Run all cells three times: PHASE="known", then "calibrate", then "unknown". Save notebooks before calibrating. Restart kernel between phases. No optimization occurs here.

In [1]:
PHASE='unknown' # known -> calibrate -> unknown
CONFIRM_ALL_DECISIONS_FIXED=True # set True for unknown phase

In [2]:
from pathlib import Path
import json, hashlib, random, math, shutil
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader,Subset
from torchvision import datasets,models,transforms
ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'configs/vanilla.yaml').exists()),None)
if ROOT is None:
    candidate=Path.cwd()/'Task 4/task4'
    if not candidate.exists():candidate=Path.cwd()/'task4'
    if not candidate.exists():raise RuntimeError('Open Jupyter in Task 4/task4.')
    ROOT=candidate.resolve()
DATA_ROOT=ROOT.parent.parent/'data'
DOWNLOAD=True # Downloads missing datasets only when YOU run a notebook.
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def read_json(p):return json.loads(Path(p).read_text(encoding='utf-8'))
def write_json(p,value):
    p=Path(p);p.parent.mkdir(parents=True,exist_ok=True)
    p.write_text(json.dumps(value,indent=2,allow_nan=False),encoding='utf-8')
def sha(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)
    return h.hexdigest()
def use(relative):
    path=ROOT/relative
    notebook=read_json(path)
    for i,cell in enumerate(notebook['cells']):
        if cell['cell_type']=='code':exec(compile(''.join(cell['source']),str(path)+f':cell-{i+1}','exec'),globals())
def seed_all():
    random.seed(6304);np.random.seed(6304);torch.manual_seed(6304);torch.cuda.manual_seed_all(6304)
    torch.backends.cudnn.benchmark=False;torch.backends.cudnn.deterministic=True
for p in ['data/make_splits.ipynb','data/cifar10.ipynb','models/resnet_cifar.ipynb','methods/vanilla.ipynb','methods/gcsc.ipynb','methods/manifold_mixup.ipynb','methods/proser.ipynb']:use(p)
print('Root:',ROOT,'Device:',DEVICE)


for p in ['scores/msp.ipynb','scores/mls.ipynb','scores/energy.ipynb','scores/mahalanobis.ipynb','evaluation/thresholds.ipynb','data/cifar100_unknowns.ipynb']:use(p)
METHODS=['vanilla','gcsc','proser']

def selected_model(method):
    path=ROOT/'results'/method/'best.pt'
    complete=ROOT/'results'/method/'complete.json'
    cfg=read_json(ROOT/'configs'/f'{method}.yaml')
    if not complete.exists() or read_json(complete)['epochs_completed']!=cfg['epochs']:
        raise RuntimeError(f'Finish all {cfg["epochs"]} epochs for {method} before extracting outputs.')
    state=torch.load(path,map_location='cpu',weights_only=False)
    for key in ['seed','epochs','batch_size','lr','momentum','weight_decay','normalization_mean','normalization_std']:
        if state['config'][key]!=cfg[key]:raise ValueError(f'{method}: checkpoint config mismatch: {key}')
    if state['split']!=read_json(ROOT/'data/split.json'):raise ValueError('Checkpoint split mismatch.')
    if method=='proser' and state['config']['vanilla_checkpoint_sha256']!=sha(ROOT/'results/vanilla/best.pt'):
        raise ValueError('PROSER was trained from a different Vanilla checkpoint.')
    model=make_model(5 if method=='proser' else 0)
    model.load_state_dict(state['model_state']);model.to(DEVICE);model.eval()
    for p in model.parameters():p.requires_grad_(False)
    return model

def cache_path(method,part):return ROOT/'cache'/method/f'{part}.npz'

def read_cache(method,part):
    path=cache_path(method,part)
    with np.load(path,allow_pickle=False) as a:out={k:a[k] for k in a.files}
    if str(out['checkpoint_sha256'].item())!=sha(ROOT/'results'/method/'best.pt'):
        raise ValueError(f'Stale cache: {path}')
    for key in ['features','logits','dummy_logits']:
        if not np.isfinite(out[key]).all():raise ValueError(f'Nonfinite {key}: {path}')
    n=len(out['labels'])
    if out['features'].shape!=(n,512) or out['logits'].shape!=(n,10) or out['dummy_logits'].shape!=(n,5 if method=='proser' else 0):
        raise ValueError(f'Unexpected output shapes: {path}')
    expected_n={'train':45000,'validation':5000,'test':10000,'near':800,'far':800}[part]
    if n!=expected_n or len(np.unique(out['indices']))!=n:raise ValueError(f'Wrong example count or duplicate indices: {path}')
    return out

@torch.no_grad()
def extract_part(model,method,part,ds,indices):
    path=cache_path(method,part);path.parent.mkdir(parents=True,exist_ok=True)
    digest=sha(ROOT/'results'/method/'best.pt')
    signature=hashlib.sha256(json.dumps({'split':read_json(ROOT/'data/split.json'),'normalization':read_json(ROOT/'configs/vanilla.yaml'),'dataset':type(ds).__name__,'train':ds.train},sort_keys=True).encode()).hexdigest()
    if path.exists():
        old=read_cache(method,part)
        if not np.array_equal(old['indices'],indices) or str(old['protocol_signature'].item())!=signature:
            raise ValueError('Existing cache has different indices or preprocessing.')
        print('Reusing',path.relative_to(ROOT));return
    features=[];logits=[];dummy=[];labels=[]
    for x,y in loader(ds,indices):
        f,z,d=forward_outputs(model,x.to(DEVICE))
        features.append(f.cpu().numpy());logits.append(z.cpu().numpy());dummy.append(d.cpu().numpy());labels.append(y.numpy())
    np.savez_compressed(path,features=np.concatenate(features),logits=np.concatenate(logits),dummy_logits=np.concatenate(dummy),labels=np.concatenate(labels),indices=np.asarray(indices),checkpoint_sha256=np.array(digest),protocol_signature=np.array(signature))
    print('Saved',path.relative_to(ROOT),len(indices),'examples')

def all_scores(method,a,means,variance,bias):
    scores={'mls':mls(a['logits'])}
    if method=='vanilla':scores.update(msp=msp(a['logits']),energy=energy(a['logits']),mahalanobis=mahalanobis(a['features'],means,variance))
    if method=='proser':scores['placeholder']=placeholder_score(a['logits'],a['dummy_logits'],bias)
    return scores

def extract_known():
    if (ROOT/'results/protocol_lock.json').exists():
        verify_lock();print('Known outputs already locked; nothing to change.');return
    # Require every model to be finished before proceeding toward final evaluation.
    for method in METHODS:
        if not (ROOT/'results'/method/'complete.json').exists():raise RuntimeError(f'Finish {method} training first.')
    _,clean,test,split=known_data()
    for method in METHODS:
        model=selected_model(method)
        for part,ds,ids in [('train',clean,split['train']),('validation',clean,split['validation']),('test',test,list(range(len(test))))]:
            extract_part(model,method,part,ds,ids)
        del model
        if DEVICE.type=='cuda':torch.cuda.empty_cache()

def calibrate_and_lock():
    if (ROOT/'results/protocol_lock.json').exists():
        verify_lock();print('Existing calibration verified; unchanged.');return
    for method in METHODS:
        model=selected_model(method);del model
        for part in ['train','validation','test']:read_cache(method,part)
    a=read_cache('vanilla','train');means,variance=fit_mahalanobis(a['features'],a['labels'])
    a=read_cache('proser','validation');bias=fit_placeholder_bias(a['logits'],a['dummy_logits'])
    thresholds={}
    for method in METHODS:
        a=read_cache(method,'validation')
        for name,u in all_scores(method,a,means,variance,bias).items():
            tau=calibrate_threshold(u)
            thresholds[f'{method}/{name}']=dict(threshold=tau,known_validation_acceptance=float(np.mean(u<=tau)),n=len(u))
    directory=ROOT/'results/calibration';directory.mkdir(exist_ok=True)
    np.savez_compressed(directory/'mahalanobis.npz',means=means,variance=variance)
    write_json(directory/'thresholds.json',thresholds)
    write_json(directory/'placeholder.json',dict(bias=bias,temperature=1024.,bias_quantile=.05,acceptance_quantile=.95))
    artifacts={str(p.relative_to(ROOT)):sha(p) for p in sorted((ROOT/'cache').rglob('*.npz'))}
    artifacts.update({str(p.relative_to(ROOT)):sha(p) for p in directory.iterdir() if p.is_file()})
    write_json(ROOT/'results/protocol_lock.json',dict(checkpoints={m:sha(ROOT/'results'/m/'best.pt') for m in METHODS},sources=source_fingerprints(),artifacts=artifacts,unknown_data_accessed=False,near=NEAR,far=FAR))
    print('Locked all checkpoints, code definitions, known caches and validation-only calibration. CIFAR-100 has not been opened by this workflow.')

def extract_unknown():
    if not CONFIRM_ALL_DECISIONS_FIXED:raise RuntimeError('Set CONFIRM_ALL_DECISIONS_FIXED=True only after completing training and calibration.')
    verify_lock()
    ds,groups=unknown_data()
    for method in METHODS:
        model=selected_model(method)
        for group,ids in groups.items():extract_part(model,method,group,ds,ids)
        del model
        if DEVICE.type=='cuda':torch.cuda.empty_cache()
    write_json(ROOT/'results/unknown_extraction.json',dict(files={str(cache_path(m,g).relative_to(ROOT)):sha(cache_path(m,g)) for m in METHODS for g in groups},protocol_lock_sha256=sha(ROOT/'results/protocol_lock.json')))
    print('Final unknown outputs cached. Do not revise this experiment based on them.')



Root: d:\LUMS GAMING\6. 2026 Fall\CS 6304\Programming Assignments\PA1\Task 4\task4 Device: cuda


In [3]:
{'known':extract_known,'calibrate':calibrate_and_lock,'unknown':extract_unknown}[PHASE]()

100%|██████████| 169M/169M [31:49<00:00, 88.5kB/s] 


Extracting d:\LUMS GAMING\6. 2026 Fall\CS 6304\Programming Assignments\PA1\data\cifar-100-python.tar.gz to d:\LUMS GAMING\6. 2026 Fall\CS 6304\Programming Assignments\PA1\data
Saved cache\vanilla\near.npz 800 examples
Saved cache\vanilla\far.npz 800 examples
Saved cache\gcsc\near.npz 800 examples
Saved cache\gcsc\far.npz 800 examples
Saved cache\proser\near.npz 800 examples
Saved cache\proser\far.npz 800 examples
Final unknown outputs cached. Do not revise this experiment based on them.
